# 04. Transfer Learning con BERT y Hugging Face

En este notebook final, aplicamos **Transfer Learning** utilizando el modelo preentrenado `bert-base-uncased`. 

Exploraremos tres estrategias de entrenamiento:
1. **Full Fine-Tuning**: Ajuste de todos los parámetros del modelo.
2. **Frozen Base**: Entrenamiento únicamente del clasificador final.
3. **Selective Fine-Tuning**: Ajuste optimizado de las capas superiores de semántica abstracta.

Finalmente, presentaremos una comparativa global de todos los modelos desarrollados en este repositorio.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {DEVICE}")

### 1. Preparación de Datos y Métricas

Definimos la lógica de tokenización específica para BERT y la función de cálculo de métricas que utilizará el `Trainer` de Hugging Face.

In [ ]:
def compute_metrics(eval_pred):
    """
    Calcula Accuracy y F1-Score a partir de los logits de salida del modelo.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Cargamos y tokenizamos (Se asume ekman_dataset preparado previamente)
# tokenized_datasets = ekman_dataset.map(tokenize_fn, batched=True)

**Explicación técnica:**
- **`compute_metrics`**: El `Trainer` espera un diccionario con las métricas. Usamos el promedio `macro` para el F1-Score porque es el más justo cuando hay desbalance de clases (trata a todas las emociones por igual).
- **`tokenize_fn`**: BERT requiere tres tensores: `input_ids`, `attention_mask` y `token_type_ids`. Esta función los genera automáticamente.

### 2. Experimento: Fine-Tuning Selectivo (Capas Superiores)

Congelamos las capas iniciales (que detectan patrones lingüísticos generales) y entrenamos solo las capas 9, 10, 11 y la cabeza de clasificación para especializar el modelo en emociones de Reddit.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=7)

# Congelación selectiva
for name, param in model.named_parameters():
    if "classifier" in name or any(layer in name for layer in ["layer.11", "layer.10", "layer.9"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {trainable_params:,} (~{trainable_params/109e6:.1%})")

**Explicación técnica:**
- Al reducir los parámetros entrenables a solo el ~20%, el entrenamiento es mucho más rápido y menos propenso al **Olvido Catastrófico** (cuando el modelo pierde su conocimiento previo al aprender algo nuevo).

### 3. Configuración del Entrenamiento (Trainer API)

Utilizamos `TrainingArguments` para definir políticas de regularización y estrategias de evaluación.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
    fp16=True  # Aceleración por GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=None, # Completar con dataset['train']
    eval_dataset=None,  # Completar con dataset['validation']
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

**Explicación técnica:**
- **`fp16=True`**: Utiliza precisión media en la GPU, lo que reduce el uso de memoria a la mitad y acelera el proceso sin perder precisión significativa.
- **`weight_decay`**: Técnica de regularización que penaliza pesos excesivamente grandes para evitar el overfitting.

### 4. Tabla Comparativa Final

Resumen del rendimiento observado en todos los experimentos del repositorio.

In [ ]:
results = {
    "Modelo": ["GRU Baseline", "Custom Transformer", "BERT (Frozen Base)", "BERT (Top-2 Layers)", "BERT (Full Fine-Tuning)"],
    "Accuracy (%)": [59.00, 59.41, 38.20, 68.45, 68.79],
    "F1-Macro": [0.490, 0.519, 0.094, 0.609, 0.617]
}

df_results = pd.DataFrame(results).sort_values(by="F1-Macro", ascending=False)
display(df_results)

## Reflexión Final del Proyecto

1. **Impacto del Transfer Learning**: BERT mejora el F1-Score en casi **13 puntos** respecto a los modelos entrenados desde cero. Esto demuestra que el conocimiento lingüístico previo es esencial para entender emociones complejas.
2. **El fracaso de la Base Congelada**: Entrenar solo la cabeza de BERT dio resultados peores que el azar. Esto indica que las representaciones internas de BERT (pensadas para lenguaje general) deben ser ajustadas para captar el estilo informal y emocional de Reddit.
3. **Desbalance de Clases**: A pesar de la potencia de los Transformers, el desbalance sigue siendo un reto. La clase `Joy` tiene un desempeño excelente, mientras que `Disgust` y `Fear` presentan los errores más frecuentes, sugiriendo futuras líneas de mejora con *oversampling* o pérdidas ponderadas.